<a href="https://colab.research.google.com/github/harnepal-hub/ai-trading-bot/blob/main/AI_trading_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ccxt pandas pandas-ta

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
PAIRS = ["B-BTC_USDT", "B-ETH_USDT", "B-SOL_USDT"]
INITIAL_CAPITAL = 1200.00

def fetch_live_data(pair, interval, limit=100):
    url = f"https://public.coindcx.com/market_data/candles?pair={pair}&interval={interval}&limit={limit}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        if isinstance(data, dict): return pd.DataFrame()
        df = pd.DataFrame(data)
        df = df.sort_values(by='time').reset_index(drop=True)
        df['datetime'] = pd.to_datetime(df['time'], unit='ms')
        df.set_index(pd.DatetimeIndex(df["datetime"]), inplace=True)
        for col in ['open', 'high', 'low', 'close', 'volume']:
            df[col] = df[col].astype(float)
        return df
    except: return pd.DataFrame()

class EvolvingPaperBot:
    def __init__(self, pairs, capital):
        self.pairs = pairs
        self.balance = capital
        self.positions = {pair: None for pair in pairs}

        # 🧠 The AI DNA (Independent for each coin)
        # Starting Bollinger Band Standard Deviation is 2.0
        self.dna_bb_std = {pair: 2.0 for pair in pairs}

    def run_cycle(self):
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n[{current_time}] 👁️ AI SCANNING MARKET (Bal: ${self.balance:,.2f})")
        print("-" * 50)

        for pair in self.pairs:
            # 1. Fetch live data
            df_1h = fetch_live_data(pair, "1h", 100)
            time.sleep(1)
            df_15m = fetch_live_data(pair, "15m", 100)
            time.sleep(1)

            if df_1h.empty or df_15m.empty:
                continue

            # 2. Calculate Regime (1H)
            df_1h['EMA_20'] = df_1h['close'].ewm(span=20, adjust=False).mean()
            df_1h['EMA_50'] = df_1h['close'].ewm(span=50, adjust=False).mean()
            df_1h['ema_spread'] = (df_1h['EMA_20'] - df_1h['EMA_50']).abs() / df_1h['close'] * 100

            latest_1h = df_1h.iloc[-2] # Look at the last fully CLOSED 1H candle
            is_bull = (latest_1h['EMA_20'] > latest_1h['EMA_50']) and (latest_1h['ema_spread'] > 0.15)
            is_bear = (latest_1h['EMA_20'] < latest_1h['EMA_50']) and (latest_1h['ema_spread'] > 0.15)

            # 3. Calculate Execution (15m) using the AI's current DNA
            std_dev = self.dna_bb_std[pair]
            df_15m['SMA_20'] = df_15m['close'].rolling(window=20).mean()
            df_15m['STD_20'] = df_15m['close'].rolling(window=20).std()
            df_15m['BBL'] = df_15m['SMA_20'] - (df_15m['STD_20'] * std_dev)
            df_15m['BBU'] = df_15m['SMA_20'] + (df_15m['STD_20'] * std_dev)

            # ATR
            high_low = df_15m['high'] - df_15m['low']
            high_close = (df_15m['high'] - df_15m['close'].shift()).abs()
            low_close = (df_15m['low'] - df_15m['close'].shift()).abs()
            tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
            df_15m['ATR'] = tr.rolling(window=14).mean()

            live_candle = df_15m.iloc[-1]
            close = live_candle['close']
            atr = live_candle['ATR']

            # 4. Trade Management
            if self.positions[pair] is not None:
                pos = self.positions[pair]
                side = pos['side']
                ep = pos['entry']
                sl = pos['sl']
                tp = pos['tp']
                size = pos['size']

                # Dynamic Breakeven Logic
                if side == 'LONG' and not pos['be_moved'] and live_candle['high'] >= pos['be_trigger']:
                    self.positions[pair]['sl'] = ep
                    self.positions[pair]['be_moved'] = True
                    print(f"🛡️ {pair}: Stop Loss moved to Breakeven!")
                elif side == 'SHORT' and not pos['be_moved'] and live_candle['low'] <= pos['be_trigger']:
                    self.positions[pair]['sl'] = ep
                    self.positions[pair]['be_moved'] = True
                    print(f"🛡️ {pair}: Stop Loss moved to Breakeven!")

                # Floating PnL & Exits
                if side == 'LONG':
                    pnl = (close - ep) * size
                    print(f"⏳ Holding {pair} LONG | Price: ${close:,.2f} | Open PnL: ${pnl:,.2f}")
                    if live_candle['low'] <= sl: self.close_trade(pair, sl, "Stop Loss", pnl > 0, pos['be_moved'])
                    elif live_candle['high'] >= tp: self.close_trade(pair, tp, "Take Profit", True, pos['be_moved'])
                else:
                    pnl = (ep - close) * size
                    print(f"⏳ Holding {pair} SHORT | Price: ${close:,.2f} | Open PnL: ${pnl:,.2f}")
                    if live_candle['high'] >= sl: self.close_trade(pair, sl, "Stop Loss", pnl > 0, pos['be_moved'])
                    elif live_candle['low'] <= tp: self.close_trade(pair, tp, "Take Profit", True, pos['be_moved'])
                continue

            # 5. Entry Logic
            print(f"🔎 {pair} | Trend: {'BULL' if is_bull else 'BEAR' if is_bear else 'CHOP'} | DNA Stretch: {std_dev:.2f} | Price: ${close:,.2f}")
            risk_amt = self.balance * 0.01

            if is_bull and close <= live_candle['BBL']:
                sl = close - (atr * 2.0)
                tp = close + (atr * 4.0)
                self.open_trade(pair, 'LONG', close, sl, tp, risk_amt / (close - sl), close + (atr * 2.0))
            elif is_bear and close >= live_candle['BBU']:
                sl = close + (atr * 2.0)
                tp = close - (atr * 4.0)
                self.open_trade(pair, 'SHORT', close, sl, tp, risk_amt / (sl - close), close - (atr * 2.0))

    def open_trade(self, pair, side, ep, sl, tp, size, be_trigger):
        self.positions[pair] = {'side': side, 'entry': ep, 'sl': sl, 'tp': tp, 'size': size, 'be_trigger': be_trigger, 'be_moved': False}
        print(f"\n🚀 [ENTRY] {pair} {side} @ ${ep:,.2f} | SL: ${sl:,.2f} | TP: ${tp:,.2f}")

    def close_trade(self, pair, exit_price, reason, is_win, is_be):
        pos = self.positions[pair]
        gross_pnl = (exit_price - pos['entry']) * pos['size'] if pos['side'] == 'LONG' else (pos['entry'] - exit_price) * pos['size']
        net_pnl = gross_pnl - ((exit_price * pos['size']) * 0.002) # Deduct 0.1% x2 fee
        self.balance += net_pnl

        print(f"\n✅ [EXIT] {pair} closed at ${exit_price:,.2f} ({reason}) | Net PnL: ${net_pnl:,.2f}")

        # 🧠 EVOLUTION ENGINE
        if is_be:
            print(f"🧠 {pair} DNA unchanged (Breakeven trade).")
        elif is_win:
            self.dna_bb_std[pair] = max(1.7, self.dna_bb_std[pair] - 0.1)
            print(f"🧠 AI EVOLVED: {pair} market is clean. Lowered BB stretch to {self.dna_bb_std[pair]:.2f} to catch more trades.")
        else:
            self.dna_bb_std[pair] = min(2.8, self.dna_bb_std[pair] + 0.2)
            print(f"🧠 AI EVOLVED: {pair} market is volatile. Raised BB stretch to {self.dna_bb_std[pair]:.2f} to demand deeper pullbacks.")

        self.positions[pair] = None

if __name__ == "__main__":
    print("==================================================")
    print("🤖 STARTING LIVE EVOLUTIONARY PAPER TRADER")
    print(f"💰 Initial Capital: ${INITIAL_CAPITAL:,.2f} USDT")
    print("==================================================\n")

    bot = EvolvingPaperBot(PAIRS, INITIAL_CAPITAL)

    while True:
        try:
            bot.run_cycle()
            # Wait 60 seconds before checking the 15m candle again
            time.sleep(60)
        except KeyboardInterrupt:
            print(f"\n🛑 Bot Stopped. Final Balance: ${bot.balance:,.2f}")
            break
        except Exception as e:
            print(f"⚠️ API Timeout, retrying in 60s...")
            time.sleep(60)

🤖 STARTING LIVE EVOLUTIONARY PAPER TRADER
💰 Initial Capital: $1,200.00 USDT


[2026-09-12 16:16:28] 👁️ AI SCANNING MARKET (Bal: $1,200.00)
--------------------------------------------------
🔎 B-BTC_USDT | Trend: CHOP | DNA Stretch: 2.00 | Price: $77,366.86
🔎 B-ETH_USDT | Trend: BULL | DNA Stretch: 2.00 | Price: $2,535.45
